In [1]:
# 1

import numpy as np
import pandas as pd

SEED = 20260810
RNG  = np.random.default_rng(SEED)

START      = "2006-01-02"
N_YEARS    = 20
TRAIN_END  = pd.Timestamp("2015-12-31")

dates = pd.bdate_range(start=START, periods=N_YEARS * 261)
dates = dates[dates.year < pd.Timestamp(START).year + N_YEARS]
T = len(dates)

VOL_LR_ANN   = 0.138     
VOL_HALFLIFE = 120       
VOL_OF_VOL   = 0.50     
RHO          = -0.45  
NU           = 6       
DRIFT_ANN    = 0.06

kappa  = np.log(2) / VOL_HALFLIFE
theta  = np.log(VOL_LR_ANN / np.sqrt(252))
sig_ou = VOL_OF_VOL * np.sqrt(2 * kappa)

z = RNG.standard_t(NU, size=T)
z /= np.sqrt(NU / (NU - 2))                 
w = RNG.standard_normal(T)

zc  = np.clip(z, -4, 4)
eps = RHO * zc / zc.std() + np.sqrt(1 - RHO**2) * w

logvol = np.empty(T)
logvol[0] = theta
for t in range(1, T):
    logvol[t] = logvol[t-1] + kappa * (theta - logvol[t-1]) + sig_ou * eps[t]

vol_idx = np.exp(logvol)
r_idx   = DRIFT_ANN / 252 + vol_idx * z

index_df = pd.DataFrame(
    {"_r_idx": r_idx, "_vol_idx": vol_idx},
    index=dates,
)
index_df["IDX"] = 3000 * np.exp(index_df["_r_idx"].cumsum())

In [2]:
# 2

BETA = {"e1":1.00,"e2":0.75,"e3":1.15,"e4":0.85,"e5":1.05,
        "e6":0.90,"e7":1.25,"e8":0.70,"e9":1.10,"e10":0.95}
IVOL = {"e1":0.22,"e2":0.30,"e3":0.20,"e4":0.25,"e5":0.18,
        "e6":0.23,"e7":0.21,"e8":0.27,"e9":0.19,"e10":0.24}

A3, A4   = 0.10, -0.085
PHI      = 0.030
W5, W6   = 0.70, 0.40
G0, TAU  = 0.45, 900.0
NOISE1   = 0.85
C7, C8   = 0.65, 0.55
E2_HL, E2_VOV = 90, 0.55
WIN = 5

RNG2 = np.random.default_rng(SEED + 1)
dsig = {k: v/np.sqrt(252) for k, v in IVOL.items()}

def tdraw(n, nu=6):
    x = RNG2.standard_t(nu, size=n)
    return x/np.sqrt(nu/(nu-2))

u = {}
for k in ["e3","e4","e5","e6","e9","e10"]:
    u[k] = dsig[k]*tdraw(T)

k2   = np.log(2)/E2_HL
s2   = E2_VOV*np.sqrt(2*k2)
th2  = np.log(dsig["e2"]) - 0.5*E2_VOV**2
lv2  = np.empty(T); lv2[0] = th2
for t in range(1, T):
    lv2[t] = lv2[t-1] + k2*(th2 - lv2[t-1]) + s2*RNG2.standard_normal()
vol_e2 = np.exp(lv2)
u["e2"] = vol_e2*tdraw(T)

u["e7"] = C7*(dsig["e7"]/dsig["e3"])*u["e3"] + np.sqrt(1-C7**2)*dsig["e7"]*tdraw(T)
u["e8"] = C8*(dsig["e8"]/dsig["e4"])*u["e4"] + np.sqrt(1-C8**2)*dsig["e8"]*tdraw(T)

roll = lambda x: pd.Series(x).rolling(WIN).sum().fillna(0.0).values
R3, R4, R10 = roll(u["e3"]), roll(u["e4"]), roll(u["e10"])
C5, C6 = np.cumsum(u["e5"]), np.cumsum(u["e6"])
g = G0*np.exp(-np.arange(T)/TAU)

u1 = np.zeros(T); C1 = np.zeros(T); S = np.zeros(T)
n1 = NOISE1*dsig["e1"]*tdraw(T)
for t in range(1, T):
    S[t-1] = C1[t-1] - W5*C5[t-1] - W6*C6[t-1]
    u1[t]  = A3*R3[t-1] + A4*R4[t-1] - PHI*S[t-1] + g[t]*R10[t-1] + n1[t]
    C1[t]  = C1[t-1] + u1[t]
S[T-1] = C1[T-1] - W5*C5[T-1] - W6*C6[T-1]
u["e1"] = u1

stock_ret = pd.DataFrame(
    {k: BETA[k]*index_df["_r_idx"].values + u[k] for k in BETA},
    index=dates,
)
idio = pd.DataFrame(u, index=dates)
spread = pd.Series(S, index=dates)

In [3]:
# 3

P0 = {"e1":48.,"e2":31.,"e3":72.,"e4":25.,"e5":96.,
      "e6":54.,"e7":18.,"e8":63.,"e9":41.,"e10":87.}

ETFW = {"e3":0.18,"e4":0.15,"e5":0.14,"e6":0.13,
        "e7":0.12,"e8":0.11,"e9":0.09,"e10":0.08}

GAP_SD, GAP_HL = 0.025, 6
ETF0 = 120.0

px = pd.DataFrame({k: P0[k]*np.exp(stock_ret[k].cumsum()) for k in P0},
                  index=dates)

RNG3 = np.random.default_rng(SEED + 2)
kg = np.log(2)/GAP_HL
sg = GAP_SD*np.sqrt(2*kg)
xg = np.zeros(T)
for t in range(1, T):
    xg[t] = xg[t-1] - kg*xg[t-1] + sg*RNG3.standard_normal()
gap = pd.Series(xg, index=dates)

basket_ret = sum(ETFW[k]*stock_ret[k] for k in ETFW)
etf_ret    = basket_ret + gap.diff().fillna(0.0)
etf_px     = ETF0*np.exp(etf_ret.cumsum())

In [4]:
# 4

from scipy.stats import norm

RF         = 0.02
IV_LOOKFWD = 21
IV_VRP     = 1.10
IV_NOISE   = 0.15
RNG4       = np.random.default_rng(SEED + 3)

exp_dates = []
for y in range(dates[0].year, dates[-1].year + 2):
    for mth in (3, 6, 9, 12):
        f = [d for d in pd.bdate_range(f"{y}-{mth}-15", f"{y}-{mth}-21") if d.weekday() == 4]
        if f and f[0] >= dates[0]:
            exp_dates.append(f[0])
exp_dates = sorted(set(exp_dates))

S2 = px["e2"].values
K  = np.zeros(T)
TT = np.zeros(T)
cur = -1
for t in range(T):
    c = 0
    while dates[t] >= exp_dates[c]:
        c += 1
    if c != cur:
        K[t] = np.round(S2[t]*2)/2
        cur = c
    else:
        K[t] = K[t-1]
    TT[t] = np.busday_count(dates[t].date(), exp_dates[c].date())/252.0

vol_tot = np.sqrt((BETA["e2"]*index_df["_vol_idx"].values)**2 + vol_e2**2)
fwd_vol = (pd.Series(vol_tot).rolling(IV_LOOKFWD).mean()
           .shift(-IV_LOOKFWD + 1).bfill().ffill().values)
opt_iv = (fwd_vol*np.sqrt(252)*IV_VRP
          * np.exp(IV_NOISE*RNG4.standard_normal(T) - 0.5*IV_NOISE**2))

d1 = (np.log(S2/K) + (RF + 0.5*opt_iv**2)*TT)/(opt_iv*np.sqrt(TT))
d2 = d1 - opt_iv*np.sqrt(TT)
opt_call = S2*norm.cdf(d1) - K*np.exp(-RF*TT)*norm.cdf(d2)
opt_put  = opt_call - S2 + K*np.exp(-RF*TT)
opt_call = np.maximum(np.round(opt_call, 2), 0.01)
opt_put  = np.maximum(np.round(opt_put,  2), 0.01)

option = pd.DataFrame({"OPT_K": K,
                       "OPT_T": np.round(TT*252).astype(int),
                       "OPT_CALL": opt_call,
                       "OPT_PUT": opt_put,
                       "OPT_IV": np.round(opt_iv, 4)}, index=dates)

In [5]:
# 5 -- export

data = pd.DataFrame(index=dates)
data.index.name = "Date"
data["IDX"] = index_df["IDX"].round(2)
for k in BETA:
    data[k] = px[k].round(2)
data["ETF"] = etf_px.round(2)
for c in option.columns:
    data[c] = option[c]

price_cols = [c for c in data.columns if not c.startswith("OPT_")]
assert data.notna().all().all(), "unexpected NaN"
assert not data.index.duplicated().any(), "duplicate dates"
assert (data[price_cols] > 0).all().all(), "non-positive price"
assert (data["OPT_T"] > 0).all(), "expired contract listed"

data.to_csv("data.csv")

print("data.csv", data.shape, data.index.min().date(), "->", data.index.max().date())
print(list(data.columns))
print("train ends", TRAIN_END.date())

data.csv (5218, 17) 2006-01-02 -> 2025-12-31
['IDX', 'e1', 'e2', 'e3', 'e4', 'e5', 'e6', 'e7', 'e8', 'e9', 'e10', 'ETF', 'OPT_K', 'OPT_T', 'OPT_CALL', 'OPT_PUT', 'OPT_IV']
train ends 2015-12-31
